# Tiny Transformer Classifier from Scratch
The objective of this notebook is to train a *Transformer* from scratch on a small dataset for text classification. The model will be trained directly on the *text classification* task. The main topics we will be working on:
1. Data preparation
2. Building the LLM architecture
3. Training an LLM

We start with the usual library imports.

> **Version complétée et expliquée.** Tous les `TODO` sont remplis. Les blocs « 💡 Explication » disent ce que fait chaque partie et pourquoi on a fait ces choix. Tu peux lire le notebook de haut en bas comme un cours.

### 💡 L'idée générale en 30 secondes

On veut un modèle qui lit une phrase en anglais (par ex. *"I feel so happy today"*) et qui répond par **une émotion parmi 6** (tristesse, joie, amour, colère, peur, surprise). C'est une **classification de texte**.

Le pipeline complet :

```
phrase ──► tokenizer ──► [101, 1045, 2514, ...] ──► Embedding ──► Bloc Transformer ──► vecteur du [CLS] ──► couche linéaire ──► 6 scores ──► émotion
          (texte → nombres)                    (nombres → vecteurs)   (les mots se « regardent »)                                 (un score par émotion)
```

- Le **tokenizer** découpe la phrase en morceaux (*tokens*) et remplace chaque morceau par un numéro.
- L'**embedding** transforme chaque numéro en un vecteur de 256 nombres que le modèle peut apprendre.
- Le **bloc Transformer** (le cœur du modèle) permet à chaque mot de récupérer de l'information sur les autres mots de la phrase, grâce à l'**attention**.
- La **tête de classification** transforme le résumé de la phrase en 6 scores, un par émotion. Le plus grand score donne la prédiction.

On code tout ça **à la main** avec PyTorch (sauf le tokenizer, qu'on prend tout fait) pour comprendre ce qu'il y a dans un Transformer.

In [1]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


💡 **Explication.** `device` vaut `cuda` si une carte graphique NVIDIA est disponible, sinon `cpu`. Le GPU fait les calculs matriciels beaucoup plus vite. **Attention :** il ne suffit pas de définir `device`. Il faut aussi y envoyer le modèle (`model.to(device)`) **et** les données (`tensor.to(device)`). Le notebook d'origine ne le faisait pas, donc tout tournait sur CPU. On le corrige plus bas, dans `train_epoch`, `evaluate` et à la création du modèle.

## Encoder-only architecture
Since we are training a *transformer model* for the task of text classification, an *encoder-only* transformer will suffice. This will allow us to focus on the content seen in the second lecture without having to worry about the decoder part.

Our encoder-only model will be constituted of different modules and sub-modules, namely:
1. An *embedding module*, formed by:
  - An *input embedding* sub-module.
  - A *positional encoding* sub-module.
2. A *transformer encoder block*, formed by:
  - A *mulit-head (self) attention* sub-module.
  - A couple of *Layer Norm* layers.
  - A *feed forward* sub-module.
  - *Residual connections*.
3. A *classification head*, formed by a linear layer.

Next we will implement the main building blocks of the *Transformer encoder block*.

### Multi-Head Attention
The *multi-head* attention layer takes the input and processes it in chunks of equal length through eahc of its different heads, by using the attention mechanism, i.e. computing the queries, keys and values, and the attention scores and weights from them.

**Exercise.** Implement the `MultiHeadAttention` by filling in the `TODO` flags below.

💡 **Explication : c'est quoi l'attention ?**

Prenons la phrase *"I didn't like this movie"*. Pour comprendre le mot *like*, le modèle doit voir qu'il y a *didn't* juste avant. L'attention sert exactement à ça : **chaque token va chercher de l'information chez les autres tokens**.

Pour chaque token, on calcule 3 vecteurs avec 3 couches linéaires différentes :
- **Query (Q)** : « qu'est-ce que je cherche ? »
- **Key (K)** : « qu'est-ce que je contiens ? » (l'étiquette que les autres consultent)
- **Value (V)** : « l'information que je donne si on me choisit ».

Puis :
1. **Scores** : on compare la query d'un token aux keys de tous les tokens avec un produit scalaire, $Q K^T$. Score élevé = « ce token m'intéresse ».
2. On divise par $\sqrt{d_{head}}$. Sans ça, les produits scalaires deviennent grands quand la dimension augmente, le softmax devient très « pointu » et les gradients deviennent minuscules.
3. **Softmax** : les scores deviennent des poids positifs qui somment à 1 (des pourcentages d'attention).
4. **Sortie** : chaque token reçoit la moyenne des values des autres, pondérée par ces poids.

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_{head}}}\right) V$$

**Pourquoi plusieurs têtes (multi-head) ?** Une seule attention ne peut regarder qu'une « chose » à la fois. Avec 8 têtes, chacune peut se spécialiser (l'une sur la négation, une autre sur le sujet du verbe, etc.). Pour que ça ne coûte pas plus cher, on **découpe** le vecteur de 256 dimensions en 8 morceaux de 32 : chaque tête travaille sur 32 dimensions. À la fin, on recolle les 8 morceaux et on les mélange avec une dernière couche linéaire `W_out`.

**Les dimensions** (b = taille du batch, s = nombre de tokens, d = hidden_dim = 256) :
- entrée `x` : `(b, s, 256)`
- après `W_query` / `W_key` / `W_value` : `(b, s, 256)`. Ce sont des `nn.Linear(256, 256)`, puisqu'on garde la même taille.
- découpage en têtes : `(b, s, 8, 32)`, puis transposition en `(b, 8, s, 32)` pour que chaque tête soit traitée comme un batch séparé
- scores : `(b, 8, s, s)`, un score pour chaque paire (token i, token j)
- sortie : `(b, 8, s, 32)` → `(b, s, 256)` → `W_out` → `(b, s, 256)`

La sortie a la **même forme que l'entrée**, ce qui permettra d'additionner les deux (connexion résiduelle).

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        # On découpe hidden_dim en num_heads morceaux égaux : il faut que la division tombe juste
        # (256 / 8 = 32 OK ; 256 / 7 ne marcherait pas).
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads  # dimension traitée par chaque tête (256 // 8 = 32)

        # Trois projections apprises : x (b, s, hidden_dim) -> Q, K, V (b, s, hidden_dim).
        # On calcule Q, K, V pour toutes les têtes d'un coup avec une seule couche chacune,
        # puis on les découpe en têtes dans forward (plus efficace que 8 petites couches).
        self.W_query = nn.Linear(hidden_dim, hidden_dim)
        self.W_key = nn.Linear(hidden_dim, hidden_dim)
        self.W_value = nn.Linear(hidden_dim, hidden_dim)

        # Après avoir recollé les têtes, on mélange leurs informations avec une dernière couche.
        self.W_out = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        batch_size, num_tokens, hidden_dim = x.shape
        assert hidden_dim == self.hidden_dim, f"hidden_dim must be {self.hidden_dim}"

        keys = self.W_key(x)  # Shape: (batch_size, num_tokens, hidden_dim)
        queries = self.W_query(x)  # Shape: (batch_size, num_tokens, hidden_dim)
        values = self.W_value(x)  # Shape: (batch_size, num_tokens, hidden_dim)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (batch_size, num_tokens, d_out) -> (batch_size, num_tokens, num_heads, head_dim)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (batch_size, num_tokens, num_heads, head_dim) -> (batch_size, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention)
        # einsum "bijk, bikl -> bijl" = produit matriciel Q @ K^T fait pour chaque batch b et chaque tête i
        # -> attn_scores : (batch_size, num_heads, num_tokens, num_tokens)
        attn_scores = torch.einsum("bijk, bikl -> bijl", queries, keys.transpose(2, 3))
        # softmax sur la dernière dimension : pour chaque token, ses poids sur les autres tokens somment à 1
        attn_weights = torch.softmax(attn_scores / (self.head_dim**0.5), dim=-1)

        # Attention output : moyenne pondérée des values -> (batch_size, num_heads, num_tokens, head_dim)
        attn_output = torch.einsum("bijk, bikl -> bijl", attn_weights, values)

        # Transpose back: (batch_size, num_heads, num_tokens, head_dim) -> (batch_size, num_tokens, num_heads, head_dim)
        attn_output = attn_output.transpose(1, 2)

        # Concatenate heads: (batch_size, num_tokens, num_heads, head_dim) -> (batch_size, num_tokens, hiddend_dim)
        attn_output = attn_output.reshape(batch_size, num_tokens, self.hidden_dim)

        # Compute output
        output = self.W_out(attn_output)

        return output

✅ **Petit test.** On vérifie que la sortie a bien la même forme que l'entrée. C'est un réflexe utile : la plupart des bugs dans un Transformer sont des erreurs de dimensions.

In [3]:
mha = MultiHeadAttention(hidden_dim=256, num_heads=8)
x_test = torch.randn(2, 10, 256)  # 2 phrases de 10 tokens, 256 dimensions par token
print(mha(x_test).shape)  # attendu : torch.Size([2, 10, 256])

torch.Size([2, 10, 256])


In [4]:
# %load solutions_nb3/mha.py

### Layer Norm
You might have heard of *BatchNorm*, where the inputs of a layer are normalized across the batch: the mean and standard deviation of the inputs are computed accross the batch dimension, then the inputs are normalized by substracting the mean and dividing by the standard deviation.

The *LayerNorm* normalization technique is similar, only the mean and standard deviation are computed accross the *feature* dimension rather than the *batch* dimension.

Once the feature mean $\mu$ and standard deviation $\sigma$ are computed, the inputs are normalized as follows:
$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}},$$
where $\epsilon$ is a small constant typically taken to be equal to $10^{-5}$ to avoid division by numbers close to zero.

The output of the `LayerNorm` is not $\hat{x}_i$ though, but
$$y_i = \gamma \hat{x}_i + \beta$$
where $\gamma$ (scale parameter) and $\beta$ (shift parameter) are learnable parameters.

**Exercise.** Complete the `LayerNorm` class below by computing the feature mean and variance and performing the appropriate normalization.

💡 **Explication : pourquoi normaliser ?**

Au fil des couches, les valeurs des vecteurs peuvent devenir très grandes ou très petites. L'entraînement devient alors instable : les gradients explosent ou s'écrasent. La normalisation remet chaque vecteur sur une échelle standard (moyenne 0, variance 1).

**Différence avec BatchNorm :**
- *BatchNorm* calcule la moyenne sur **les exemples du batch**. Pour du texte, c'est gênant : les phrases ont des longueurs différentes (avec du padding), et le résultat dépendrait des autres phrases du batch.
- *LayerNorm* calcule la moyenne sur **les 256 nombres d'un même token**. Chaque token est normalisé tout seul, indépendamment du batch. C'est pour ça que tous les Transformers l'utilisent.

**Choix d'implémentation :**
- `dim=-1` : la dernière dimension, celle des features (les 256 nombres).
- `keepdim=True` : garde la forme `(b, s, 1)` au lieu de `(b, s)`, pour que la soustraction `x - mean` fonctionne par *broadcasting* (PyTorch répète automatiquement la moyenne sur les 256 colonnes).
- `unbiased=False` : la variance « de population » (on divise par N et non par N−1). C'est la convention de la formule de l'énoncé et de `nn.LayerNorm`.
- `scale` (γ) et `shift` (β) sont appris. Ils permettent au modèle de « défaire » la normalisation si c'est utile. Au départ, γ=1 et β=0, donc on ne change rien.

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.eps = 1e-5 # small value to avoid division by zero
        self.scale = nn.Parameter(torch.ones(hidden_dim)) # scale parameter (learnable)
        self.shift = nn.Parameter(torch.zeros(hidden_dim)) # shift parameter (learnable)

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)  # moyenne des features de chaque token : (b, s, 1)
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # variance des features de chaque token : (b, s, 1)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)  # moyenne 0, variance 1
        y = self.scale * norm_x + self.shift  # gamma * x_hat + beta
        return y

✅ **Test.** On compare notre LayerNorm à celle de PyTorch (`nn.LayerNorm`). Si on a bien codé, elles donnent le même résultat.

In [6]:
x_test = torch.randn(2, 10, 256)
ours = LayerNorm(256)(x_test)
reference = nn.LayerNorm(256)(x_test)
print("Même résultat que nn.LayerNorm :", torch.allclose(ours, reference, atol=1e-5))
print("Moyenne d'un token après normalisation :", ours[0, 0].mean().item())  # ~ 0
print("Écart-type d'un token après normalisation :", ours[0, 0].std(unbiased=False).item())  # ~ 1

Même résultat que nn.LayerNorm : True
Moyenne d'un token après normalisation : -1.862645149230957e-09
Écart-type d'un token après normalisation : 0.9999950528144836


In [7]:
# %load solutions_nb3/layernorm.py

### Feed-Forward Network
The *feed forward* sub-modules in our *transformer blocks* will consist of:
1. A linear layer, the output dimension being 4 times `hidden_dim`.
2. A ReLU activation
3. A linear layer, the output dimension being `hidden_dim`.

**Exercise.** Work out the input dimension of the two linear layers, and complete the `FeedForward` class by filling in the `TODO` flags.

💡 **Explication.** L'attention **mélange l'information entre les tokens**. Le feed-forward, lui, **transforme chaque token individuellement** (le même petit réseau est appliqué à chaque position). On peut le voir comme l'endroit où le modèle « réfléchit » sur ce que chaque token a récupéré.

**Dimensions :**
- 1ʳᵉ couche : entrée `hidden_dim` (256), sortie `4 × hidden_dim` (1024). On **élargit** pour avoir plus de capacité de calcul. Le facteur 4 est la convention du papier original *Attention is all you need*.
- ReLU : la non-linéarité. Sans elle, deux couches linéaires à la suite équivaudraient à une seule couche linéaire, et on ne gagnerait rien.
- 2ᵉ couche : entrée `4 × hidden_dim` (1024), sortie `hidden_dim` (256). On **revient à la taille d'origine** pour pouvoir additionner avec l'entrée (connexion résiduelle).

`nn.Linear` s'applique sur la dernière dimension, donc `(b, s, 256) → (b, s, 1024) → (b, s, 256)` fonctionne directement.

In [8]:
class FeedForward(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(hidden_dim, 4 * hidden_dim),  # 256 -> 1024 : on élargit
            nn.ReLU(),                              # non-linéarité
            nn.Linear(4 * hidden_dim, hidden_dim),  # 1024 -> 256 : on revient à la taille d'origine
        )

    def forward(self, x):
        return self.layers(x)

In [9]:
# %load solutions_nb3/feedforward.py

We are done with the three main building blocks that form the *transformer encoder block*. The next steps are to implement the *transformer encoder block* itself as well as the *embedding sub-module*. Once those two sub-modules will be implemented, we will be ready to implement the final *end-to-end* transformer classifier model.


### Transformer Encoder Block
As we have mentioned earlier, the *Transformer encoder block* is composed of:
1. A *Multi-head attention* sub-module
2. A *LayerNorm* normalization
3. A *Feed Forward* sub-module
4. A *LayerNorm* normalization

Moreover, two skip-connections are present in the *Transformer Block*:
- *(A)* A skip connection that adds the original input to the output of the operation *1* above.
- *(B)* A skip connection that adds the output of the operation *2* to the output of the operation *3* above.

**Exercise.** Implement the `TransformerEncoderBlock` module below by completing the `TODO` tags.

💡 **Explication : assembler les briques.**

```
x ──► Attention ──► (+ x) ──► LayerNorm ──► FeedForward ──► (+) ──► LayerNorm ──► sortie
 └───────────────────┘          │                            ▲
      connexion (A)             └────────────────────────────┘
                                         connexion (B)
```

**Les connexions résiduelles (skip connections)** : au lieu de remplacer `x` par `f(x)`, on calcule `x + f(x)`. Deux raisons :
1. Le gradient peut « traverser » directement le `+` pendant la rétropropagation. On évite ainsi qu'il disparaisse dans un réseau profond.
2. La couche n'a qu'à apprendre une **correction** de `x`, ce qui est plus facile que de tout recalculer. Au début de l'entraînement, le bloc est presque l'identité, donc stable.

C'est pour ça que l'attention et le feed-forward gardent la dimension 256 : on ne peut additionner que des tenseurs de même forme.

**Choix :** on utilise **notre** classe `LayerNorm` (codée au-dessus) plutôt que `nn.LayerNorm`. Le résultat est le même (on l'a vérifié), mais l'intérêt du TP est d'utiliser nos propres briques. L'ordre suit l'énoncé : la normalisation vient **après** l'addition. C'est le « Post-LN » du papier original. Les grands modèles récents (GPT, Llama) normalisent plutôt **avant** (« Pre-LN »), parce que c'est plus stable quand on empile beaucoup de blocs. Avec un seul bloc ici, ça ne change rien.

In [10]:
class TransformerEncoderBlock(nn.Module):
  def __init__(self, hidden_dim, num_heads):
    super(TransformerEncoderBlock, self).__init__()

    self.attention = MultiHeadAttention(hidden_dim, num_heads)
    self.norm1 = LayerNorm(hidden_dim)
    self.norm2 = LayerNorm(hidden_dim)
    self.feed_forward = FeedForward(hidden_dim)

  def forward(self, x):
      # 1. attention + connexion résiduelle (A), puis normalisation
      x = self.norm1(x + self.attention(x))
      # 2. feed-forward + connexion résiduelle (B), puis normalisation
      x = self.norm2(x + self.feed_forward(x))
      return x

In [11]:
# %load solutions_nb3/encoderblock.py

### Embedding
As we have mentioned above, the *embedding sub-module* consists of both:
- An *input embedding*: taking as input a token ID given by the tokenizer
- A *positional encoding*: taking as input the position of a token in the sentence.

**Question.** What is the input dimension of the *input embedding*? What about the *positional encoding*?

**Exercise.** Implement the `Embedding` sub-module by completing the `TODO` tags below.

💡 **Explication.**

**Input embedding.** Le tokenizer donne des numéros (par ex. `happy` → 3407). Un numéro brut n'a pas de sens pour un réseau : 3407 n'est pas « plus grand » que 2000 en un sens utile. `nn.Embedding(vocab_size, hidden_dim)` est un **tableau de `vocab_size` lignes et `hidden_dim` colonnes** : le token numéro 3407 récupère la ligne 3407, un vecteur de 256 nombres appris pendant l'entraînement. Des mots de sens proche finiront avec des vecteurs proches.

**Positional encoding.** L'attention ne connaît pas l'ordre des mots : pour elle, *"dog bites man"* et *"man bites dog"* sont le même ensemble de tokens. On ajoute donc à chaque token un vecteur qui dépend de **sa position** (0, 1, 2…). On utilise aussi un `nn.Embedding`, cette fois indexé par la position. On dit que l'encodage de position est **appris**, comme dans BERT. Le papier original utilisait plutôt des sinus et cosinus fixes.

**Réponse à la question :**
- *input embedding* : l'entrée est un numéro de token entre 0 et `vocab_size - 1`. La « dimension d'entrée » est donc **`vocab_size`** (30 522 pour BERT), le nombre de lignes du tableau.
- *positional encoding* : l'entrée est une position entre 0 et `max_length - 1`. La dimension d'entrée est donc **`max_length`** (512).
- Les deux produisent des vecteurs de taille `hidden_dim`, et on les **additionne**.

**Détail technique :** `torch.arange(seq_length, device=x.device)` crée `[0, 1, …, s-1]` **sur le même device que x**. Sinon on mélangerait un tenseur GPU et un tenseur CPU, ce qui donne une erreur. Le vecteur de positions a la forme `(s, 256)`, et il est ajouté à `(b, s, 256)` par broadcasting : chaque phrase du batch reçoit les mêmes positions.

In [12]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, max_length, hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, hidden_dim)  # une ligne apprise par token du vocabulaire
    self.position_encoding = nn.Embedding(max_length, hidden_dim)  # une ligne apprise par position

  def forward(self, x):
    # x : (batch_size, seq_length) contient des numéros de tokens
    seq_length = x.shape[1]
    positions = torch.arange(seq_length, device=x.device)  # [0, 1, ..., seq_length - 1]
    token_embeddings = self.embedding(x)  # (batch_size, seq_length, hidden_dim)
    position_embeddings = self.position_encoding(positions)  # (seq_length, hidden_dim)
    return token_embeddings + position_embeddings  # broadcasting -> (batch_size, seq_length, hidden_dim)

In [13]:
# %load solutions_nb3/embedding.py

### Transformer Encoder-Only Model for Classification
We have implemented all necessary sub-modules ane we are now ready to implement the *end-to-end* architecture of the *transformer encoder-only* model. To that end, we will make use of:
- The `Embedding` sub-module
- The `TransformerEncoderBlock` sub-module
- A `nn.Linear` layer to act as a classification head.

**Note.** The output of the `TransformerEncoderBlock` sub-module is a tensor of shape $(b, s, d_h)$ where $b$ represents the batch size, $s$ the sequence length, and $d_h$ the hidden dimension of the model. In order to perform the classification task, we need to compute one logit per sequence in the batch. In order to do so, we use a strategy common to the *BERT family* of models: we only use the representation of the special token `[CLS]` as input to the classification head. Note that in order for this strategy to work, we will use a tokenizer that adds the token `[CLS]` at the beginning of each input token sequence.

**Exercise.** Implement the `TransformerClassifier` model by completing the `TODO` tags below.

💡 **Explication : pourquoi le token `[CLS]` ?**

Après l'encodeur, on a **un vecteur par token**, `(b, s, 256)`. Or on veut **une seule prédiction par phrase**. Il faut donc résumer la phrase en un seul vecteur.

L'astuce de BERT : le tokenizer ajoute un token spécial `[CLS]` (pour *classification*) **au début de chaque phrase**. Ce token ne correspond à aucun mot. Grâce à l'attention, il peut regarder tous les autres tokens, et pendant l'entraînement le modèle apprend à y concentrer le résumé utile pour la classification. On prend donc `x[:, 0, :]`, la position 0 de chaque phrase, de forme `(b, 256)`.

(Une autre option serait de faire la moyenne de tous les tokens, mais on suit l'énoncé.)

**Tête de classification :** `nn.Linear(hidden_dim, num_classes)` transforme le vecteur de 256 nombres en **6 scores**, un par émotion. On les appelle des **logits** : ce ne sont pas encore des probabilités (ils peuvent être négatifs). On n'ajoute **pas** de softmax ici, parce que la fonction de perte `CrossEntropyLoss` le fait elle-même, de façon plus stable numériquement.

In [14]:
class TransformerClassifier(nn.Module):
  def __init__(self,
               vocab_size,
               max_length,
               hidden_dim,
               num_heads,
               num_classes):
    super().__init__()

    self.embedding = Embedding(vocab_size, max_length, hidden_dim)
    self.encoder = TransformerEncoderBlock(hidden_dim, num_heads)
    self.classifier_head = nn.Linear(hidden_dim, num_classes)

  def forward(self, x):
    x = self.embedding(x)  # (b, s) -> (b, s, hidden_dim)
    x = self.encoder(x)  # (b, s, hidden_dim) -> (b, s, hidden_dim)
    x = x[:, 0, :] # We only use the encoding of the token [CLS] as input of the classification head.
    x = self.classifier_head(x)  # (b, hidden_dim) -> (b, num_classes) : les logits
    return x

In [15]:
# %load solutions_nb3/classifier.py

## Data Preparation
We will train the `TransformerClassifier` model on a text classification task. The data set we will be using is the following [dataset](https://huggingface.co/datasets/dair-ai/emotion/viewer). It consists of sentences grouped into 6 different classes according to the main emotion they convey:

| Class ID | Emotion |
| -------- | ------- |
| 0 | sadness |
| 1 | joy |
| 2 | love |
| 3 | anger |
| 4 | fear |
| 5 | surprise |



**Questions.** Answer the following questions:
1. What type of object is the ``raw_dataset`` object ?
2. How many elements are there in the ``raw_dataset`` object ?
3. What type of object is the ``raw_dataset["train"]`` object ?
4. Describe the ``raw_dataset["train"]`` object.
5. Are the train, validation and test datasets balanced ?

**Exercise.** Print one of the elements of ``raw_dataset["train"]``.

In [16]:
from datasets import load_dataset

raw_dataset = load_dataset("dair-ai/emotion")
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

💡 **Réponses aux questions.**

1. `raw_dataset` est un **`DatasetDict`** (bibliothèque 🤗 `datasets`) : une sorte de dictionnaire Python dont les clés sont les découpages du jeu de données.
2. Il contient **3 éléments** : `train` (16 000 phrases), `validation` (2 000) et `test` (2 000).
3. `raw_dataset["train"]` est un **`Dataset`** : un tableau qu'on peut indexer comme une liste (`raw_dataset["train"][0]`).
4. Il a **2 colonnes** : `text` (la phrase, en minuscules) et `label` (un entier de 0 à 5 qui code l'émotion).
5. **Non, les données ne sont pas équilibrées** (voir le comptage plus bas). Il y a environ un tiers de *joy* et de *sadness*, mais seulement ~3-4 % de *surprise*. Conséquence : un modèle qui répondrait toujours *joy* aurait déjà ~35 % de bonnes réponses. Il faut garder ça en tête pour juger l'accuracy, et le modèle aura plus de mal sur les classes rares. Les trois découpages ont à peu près les mêmes proportions.

**Vocabulaire :**
- *train* sert à **apprendre** les poids.
- *validation* sert à **surveiller** l'entraînement (détecter le sur-apprentissage, choisir les hyperparamètres).
- *test* sert à donner **une seule fois, à la fin,** une estimation honnête de la performance sur des données jamais vues.

In [17]:
print(raw_dataset["train"][0])  # un exemple : {'text': ..., 'label': ...}

label_names = raw_dataset["train"].features["label"].names  # ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
print("Exemple :", raw_dataset["train"][0]["text"], "->", label_names[raw_dataset["train"][0]["label"]])

{'text': 'i didnt feel humiliated', 'label': 0}
Exemple : i didnt feel humiliated -> sadness


Vérification de l'équilibre des classes (question 5) : on compte le pourcentage de chaque émotion dans chaque découpage.

In [18]:
import pandas as pd

proportions = {
    split: pd.Series(raw_dataset[split]["label"]).value_counts(normalize=True).sort_index()
    for split in ["train", "validation", "test"]
}
proportions = pd.DataFrame(proportions)
proportions.index = label_names
print((proportions * 100).round(1))  # en %

          train  validation  test
sadness    29.2        27.5  29.0
joy        33.5        35.2  34.8
love        8.2         8.9   8.0
anger      13.5        13.8  13.8
fear       12.1        10.6  11.2
surprise    3.6         4.0   3.3


### Pre-trained Tokenizer
In order to convert the textual data in the above datasets into a format that can be used as input for our models, we first *tokenize* the text using a pre-trained tokenizer.

In [19]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

💡 **Explication.** On ne code pas le tokenizer nous-mêmes : on récupère celui de **BERT** (`bert-base-uncased`), déjà entraîné sur beaucoup de texte anglais. *Uncased* veut dire qu'il met tout en minuscules.

C'est un tokenizer **WordPiece** : les mots fréquents sont un seul token (`happy`), les mots rares sont découpés en morceaux (`unbelievable` → `un`, `##bel`, `##ie`, `##vable`). Ça permet de représenter n'importe quel mot avec un vocabulaire limité (~30 000 tokens).

⚠️ On prend **seulement le tokenizer**, pas le modèle BERT : les poids de notre Transformer partent de zéro (aléatoires).

**Exercise.** Let's find out more about the tokenizer we will be using. Write code in order to answer the following questions:
1. What is the name of the tokenizer being used?
2. What is the size of the vocabulary?
3. What is the maximum model input length?
4. What special tokens does the tokenizer use? What are their IDs?

**Remark.** Check that the special token `[CLS]` is indeed one of the special tokens in the pre-trained tokenizer. What is its token ID?

In [20]:
print("1. Nom du tokenizer :", tokenizer.name_or_path, "|", type(tokenizer).__name__)
print("2. Taille du vocabulaire :", tokenizer.vocab_size)
print("3. Longueur max d'une entrée :", tokenizer.model_max_length)
print("4. Tokens spéciaux et leurs IDs :")
for token in tokenizer.all_special_tokens:
    print(f"   {token:7s} -> {tokenizer.convert_tokens_to_ids(token)}")

print()
print("[CLS] est bien un token spécial :", "[CLS]" in tokenizer.all_special_tokens,
      "| son ID :", tokenizer.cls_token_id)

# Exemple concret : le tokenizer ajoute [CLS] au début et [SEP] à la fin
encoded = tokenizer("I feel so happy today")
print()
print("IDs    :", encoded["input_ids"])
print("Tokens :", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

1. Nom du tokenizer : bert-base-uncased | BertTokenizer
2. Taille du vocabulaire : 30522
3. Longueur max d'une entrée : 512
4. Tokens spéciaux et leurs IDs :
   [UNK]   -> 100
   [SEP]   -> 102
   [PAD]   -> 0
   [CLS]   -> 101
   [MASK]  -> 103

[CLS] est bien un token spécial : True | son ID : 101

IDs    : [101, 1045, 2514, 2061, 3407, 2651, 102]
Tokens : ['[CLS]', 'i', 'feel', 'so', 'happy', 'today', '[SEP]']


💡 **Ce qu'il faut retenir :**
- vocabulaire de **30 522** tokens : ce sera `VOCAB_SIZE`, le nombre de lignes de notre `nn.Embedding` ;
- longueur max **512** tokens : ce sera `MAX_LENGTH`, le nombre de positions possibles ;
- tokens spéciaux :
  - `[PAD]` = 0 : *padding*, pour compléter les phrases courtes afin que toutes celles d'un batch aient la même longueur ;
  - `[UNK]` = 100 : token inconnu ;
  - `[CLS]` = 101 : ajouté **au début** de chaque phrase, c'est lui qu'on utilise pour classer ;
  - `[SEP]` = 102 : ajouté à la fin, il sépare deux phrases ;
  - `[MASK]` = 103 : utilisé pour pré-entraîner BERT, pas ici.

Comme on le voit dans l'exemple, `[CLS]` est **toujours en position 0**. C'est ce qui rend valide `x[:, 0, :]` dans notre classifieur.

### Tokenizing the raw input
We next use the pre-trained tokenizer to tokenize the whole raw dataset:

💡 `map(..., batched=True)` applique le tokenizer à toutes les phrases (par paquets, pour aller vite) et ajoute les colonnes `input_ids` et `attention_mask`. `truncation=True` coupe les phrases de plus de 512 tokens : sinon il n'y aurait pas de ligne dans `position_encoding` pour ces positions, et on aurait une erreur. `set_format("torch", ...)` fait renvoyer des tenseurs PyTorch et ne garde que les colonnes dont on a besoin.

In [21]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)


tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format("torch", columns=["input_ids", "label"])
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

### Data Loaders
Finally, we create three separate pytorch data loaders in order to easily iterate through them during the training and evaluation phases.

In [22]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = torch.utils.data.DataLoader(
    tokenized_dataset["train"],
    shuffle=True,
    batch_size=8,
    collate_fn=data_collator
)

val_loader = torch.utils.data.DataLoader(
    tokenized_dataset["validation"],
    shuffle=False,
    batch_size=8,
    collate_fn=data_collator
)

test_loader = torch.utils.data.DataLoader(
    tokenized_dataset["test"],
    shuffle=False,
    batch_size=8,
    collate_fn=data_collator
)

💡 **Explication.**
- Un **DataLoader** découpe le dataset en **batchs** (ici 8 phrases) et les donne un par un à la boucle d'entraînement.
- `shuffle=True` pour le train seulement : on mélange à chaque epoch pour que le modèle ne voie pas toujours les exemples dans le même ordre. Pour validation et test, l'ordre n'a pas d'importance.
- Les phrases d'un batch n'ont pas la même longueur. Or un tenseur doit être rectangulaire. Le **`DataCollatorWithPadding`** complète chaque phrase avec des `[PAD]` (ID 0) jusqu'à la longueur de la plus longue phrase **du batch**. Ce *padding dynamique* fait moins de calculs inutiles que de tout compléter à 512.
- Le collator **renomme `label` en `labels`**, d'où `batch['labels']` dans la boucle d'entraînement.

⚠️ **Limite de notre modèle :** notre attention ne reçoit pas l'`attention_mask`, donc les tokens réels « regardent » aussi les `[PAD]`. Ça ajoute un peu de bruit, mais l'entraînement fonctionne quand même. Un vrai Transformer mettrait les scores des positions `[PAD]` à −∞ avant le softmax, pour qu'elles reçoivent un poids nul.

## Training and Evaluation
Now that we have both our model architecture defined and our data prepared, we can proceed to the last phase of the lab project: training and evaluating the model.

### Initialize model
We start by setting all the necessary arguments in order to instantiate the concrete model that we will be using for the classification task.

**Exercise.** Set all the necessary arguments:
- `VOCAB_SIZE`: the size of the vocabulary of the pre-trained tokenizer.
- `MAX_LENGTH`: the maximum length of the token sequence of the pre-trained tokenizer.
- `HIDDEN_DIM`: 256
- `NUM_HEADS`: 8
- `NUM_CLASSES`: The number of classes of the multi-class classification task.

**Question.** Make sure that the hidden dimension is divisible by the number of heads.

💡 **Justification des valeurs :**
- `VOCAB_SIZE = tokenizer.vocab_size` (30 522) : chaque ID que le tokenizer peut produire doit avoir sa ligne dans l'embedding.
- `MAX_LENGTH = tokenizer.model_max_length` (512) : à cause de `truncation=True`, aucune phrase ne dépasse 512 tokens, donc 512 positions suffisent.
- `HIDDEN_DIM = 256`, `NUM_HEADS = 8` : imposés par l'énoncé. 256 / 8 = 32, **c'est bien divisible** (le `assert` de `MultiHeadAttention` le vérifie aussi). C'est petit comparé à BERT-base (768 dimensions, 12 têtes, 12 blocs), mais suffisant pour 16 000 phrases et rapide à entraîner.
- `NUM_CLASSES` : lu dans les métadonnées du dataset (6) plutôt qu'écrit en dur. Si on change de dataset, la valeur reste juste.

In [23]:
VOCAB_SIZE = tokenizer.vocab_size
MAX_LENGTH = tokenizer.model_max_length
HIDDEN_DIM = 256
NUM_HEADS = 8
NUM_CLASSES = raw_dataset["train"].features["label"].num_classes

assert HIDDEN_DIM % NUM_HEADS == 0  # 256 / 8 = 32 dimensions par tête

print(f"VOCAB_SIZE: {VOCAB_SIZE}")
print(f"MAX_LENGTH: {MAX_LENGTH}")
print(f"HIDDEN_DIM: {HIDDEN_DIM}")
print(f"NUM_HEADS: {NUM_HEADS}")
print(f"NUM_CLASSES: {NUM_CLASSES}")

VOCAB_SIZE: 30522
MAX_LENGTH: 512
HIDDEN_DIM: 256
NUM_HEADS: 8
NUM_CLASSES: 6


In [24]:
# %load solutions_nb3/classifier_params.py

**Exercise.** Instantiate the `classifier` model using the above arguments.

In [25]:
torch.manual_seed(0)  # même initialisation aléatoire à chaque exécution -> résultats reproductibles
classifier = TransformerClassifier(VOCAB_SIZE, MAX_LENGTH, HIDDEN_DIM, NUM_HEADS, NUM_CLASSES).to(device)
print(classifier)

num_params = sum(p.numel() for p in classifier.parameters())
print(f"\nNombre de paramètres : {num_params:,}")

TransformerClassifier(
  (embedding): Embedding(
    (embedding): Embedding(30522, 256)
    (position_encoding): Embedding(512, 256)
  )
  (encoder): TransformerEncoderBlock(
    (attention): MultiHeadAttention(
      (W_query): Linear(in_features=256, out_features=256, bias=True)
      (W_key): Linear(in_features=256, out_features=256, bias=True)
      (W_value): Linear(in_features=256, out_features=256, bias=True)
      (W_out): Linear(in_features=256, out_features=256, bias=True)
    )
    (norm1): LayerNorm()
    (norm2): LayerNorm()
    (feed_forward): FeedForward(
      (layers): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
    )
  )
  (classifier_head): Linear(in_features=256, out_features=6, bias=True)
)

Nombre de paramètres : 8,736,006


💡 **Lecture du résultat.** On a environ **8,8 millions de paramètres**. La grande majorité (30 522 × 256 ≈ 7,8 M) est dans **l'embedding des tokens**. Le bloc Transformer lui-même ne fait qu'environ 0,8 M de paramètres (4 × 256² pour l'attention, 2 × 256 × 1024 pour le feed-forward). Pour comparer, BERT-base en a 110 M.

`.to(device)` envoie tous les poids sur le GPU. C'est indispensable pour profiter de `device`.

In [26]:
# %load solutions_nb3/instantiate_classifier.py

### Train Model
We can now proceed to the model training phase. In order to do so, we will define two functions:
- A `train_epoch` function that will train the model by iterating through the given data loader once.
- An `evaluate` function that will evaluate the model on the given data loader.

💡 **Ce qui se passe à chaque batch (`train_epoch`) :**
1. `optimizer.zero_grad()` : remet les gradients à zéro, parce que PyTorch les **additionne** par défaut d'un batch à l'autre.
2. `outputs = model(input_ids)` : la **passe avant**, qui calcule les logits `(8, 6)`.
3. `loss = criterion(outputs, labels)` : mesure l'erreur.
4. `loss.backward()` : la **rétropropagation**, qui calcule le gradient de la loss par rapport à chaque poids.
5. `optimizer.step()` : modifie les poids dans la direction qui réduit la loss.

L'**accuracy** est la proportion de phrases dont la classe au plus grand logit (`argmax`) est la bonne.

**Dans `evaluate` :** `model.eval()` et `torch.no_grad()` signifient qu'on ne calcule pas de gradients et qu'on ne modifie pas les poids. On ne fait que mesurer, et ça va plus vite.

**Modification par rapport à l'énoncé :** on ajoute `.to(device)` sur `input_ids` et `labels`. Le modèle est sur le GPU, donc ses entrées doivent y être aussi. Sinon on obtient l'erreur *« Expected all tensors to be on the same device »*.

In [27]:
import time
from tqdm import tqdm

In [28]:
def train_epoch(model, dataloader, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0

    model.train()

    for batch in tqdm(dataloader, desc="Processing Batches"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)  # les données doivent être sur le même device que le modèle
        labels = batch['labels'].to(device)

        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        acc = (outputs.argmax(dim=1) == labels).float().mean()

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(dataloader), epoch_acc / len(dataloader)

In [29]:
def evaluate(model, dataloader, criterion):
    epoch_loss = 0
    epoch_acc = 0
    model.eval()

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing Batches"):

            input_ids = batch["input_ids"].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids)
            loss = criterion(outputs, labels)
            acc = (outputs.argmax(dim=1) == labels).float().mean()

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(dataloader), epoch_acc / len(dataloader)

Let us proceed to the actual training of the model.

**Questions.** What loss function should we use to train the model?

**Exercise.** Train the model by:
- Setting the number of epochs to 2 and the learning rate to 0.001.
- Completing the `TODO` tags below.

💡 **Réponse : quelle fonction de perte ?** La **cross-entropy** (`nn.CrossEntropyLoss`), la perte standard de la classification **multi-classe** (une seule bonne classe parmi 6). Elle fait deux choses :
1. elle applique un **softmax** aux 6 logits pour obtenir des probabilités ;
2. elle calcule $-\log(p_{\text{bonne classe}})$.

Si le modèle donne une probabilité de 0,9 à la bonne émotion, la perte vaut 0,1. S'il lui donne 0,01, la perte vaut 4,6. Elle punit donc fortement les erreurs commises avec assurance.

Elle prend directement les **logits** et les **labels entiers** (0 à 5), d'où l'absence de softmax dans le modèle. Au départ, avec des prédictions au hasard sur 6 classes, la perte vaut environ $\ln 6 \approx 1{,}79$ : c'est le point de départ à battre.

**Autres choix :**
- **Adam** est un optimiseur qui adapte le pas d'apprentissage à chaque paramètre. C'est le choix par défaut qui marche bien sans réglage fin.
- `lr = 0.001` et 2 epochs sont imposés par l'énoncé. Une **epoch** = un passage complet sur les 16 000 phrases d'entraînement (2 000 batchs de 8).
- On affiche la **durée** de chaque epoch. Le code d'origine affichait `time.time()`, c'est-à-dire l'heure en secondes depuis 1970, ce qui n'avait pas de sens.

In [30]:
EPOCHS = 2
LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(classifier.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    start_time = time.time()

    train_loss, train_acc = train_epoch(classifier, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(classifier, val_loader, criterion)

    epoch_time = time.time() - start_time  # durée de l'epoch en secondes

    print("")
    print(f'Epoch: {epoch+1:02} | Time: {epoch_time:.0f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')


Epoch: 01 | Time: 30s
	Train Loss: 1.602 | Train Acc: 32.67%
	 Val. Loss: 1.590 |  Val. Acc: 29.40%



Epoch: 02 | Time: 29s
	Train Loss: 1.574 | Train Acc: 35.12%
	 Val. Loss: 1.559 |  Val. Acc: 38.85%


In [31]:
# %load solutions_nb3/train_classifier.py

**Question.** Is the model overfitting? Should we keep training during more epochs?

💡 **Réponse : le modèle sur-apprend-il ?**

Le **sur-apprentissage** (*overfitting*), c'est quand le modèle apprend les exemples d'entraînement par cœur au lieu d'apprendre des règles générales. On le repère en comparant train et validation : si la **loss de train baisse alors que celle de validation remonte**, et que l'écart d'accuracy entre les deux se creuse, le modèle sur-apprend.

**Ici, ce n'est pas le cas, c'est même le problème inverse : le modèle *sous-apprend* (underfitting).** Avec `lr = 0.001`, on obtient (les chiffres varient un peu d'une exécution à l'autre) :
- une loss de train et une loss de validation presque égales et qui baissent à peine (~1,60 → ~1,57) ;
- une accuracy d'environ 35-40 %, à peine mieux que de répondre toujours *joy* (~35 %).

Un calcul permet de comprendre. Si un modèle ne lit **pas du tout** la phrase et prédit juste les fréquences des classes (29 % sadness, 33 % joy, etc.), sa cross-entropy vaut l'entropie de ces fréquences : $-\sum_c p_c \ln p_c \approx 1{,}58$. **Notre loss (~1,57) est exactement à ce niveau** : le modèle a seulement appris quelles émotions sont fréquentes, pas à lire les phrases.

**Pourquoi ?** Le **learning rate de 0,001 est trop grand** pour ce modèle. Les poids bougent trop à chaque pas, et le modèle tombe dans la solution facile « prédire la classe majoritaire » sans en sortir. C'est un problème classique des Transformers : ils sont sensibles au learning rate, surtout avec la normalisation après l'addition (Post-LN) et sans *warmup* (augmentation progressive du learning rate au début).

**Faut-il entraîner plus longtemps ?** Avec ce learning rate, ce serait surtout du temps perdu : la loss stagne. La bonne réaction est de **baisser le learning rate**. C'est ce qu'on teste juste après l'évaluation sur le test set.

### Evaluation
Finally, we evaluate the model on the test set.

In [32]:
test_loss, test_acc = evaluate(classifier, test_loader, criterion)
print("")
print(f'Test Loss: {test_loss:.3f} |  Test Acc: {test_acc*100:.2f}%')


Test Loss: 1.539 |  Test Acc: 38.30%


💡 **Interprétation.** Le test set n'a **jamais** servi pendant l'entraînement : c'est l'estimation la plus honnête de la performance sur de nouvelles phrases. Repères :
- le hasard donne ~17 % (1 chance sur 6) ;
- répondre toujours *joy* donne ~35 % ;
- un BERT pré-entraîné puis fine-tuné (Notebook 2) atteint ~93 %.

Avec `lr = 0.001`, notre modèle est au niveau de « toujours répondre *joy* ». On a vu pourquoi dans la question précédente.

### Pour aller plus loin : même modèle, learning rate 10 fois plus petit

On garde **exactement** la même architecture, les mêmes données et 2 epochs. On change **une seule chose** : `lr = 0.0001` au lieu de `0.001`. Changer un seul paramètre à la fois permet d'être sûr que la différence vient de lui.

On recrée le modèle depuis zéro (nouveaux poids aléatoires, même graine), sinon on repartirait du modèle déjà entraîné et la comparaison serait faussée.

In [33]:
torch.manual_seed(0)
classifier = TransformerClassifier(VOCAB_SIZE, MAX_LENGTH, HIDDEN_DIM, NUM_HEADS, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-4)

for epoch in range(EPOCHS):
    start_time = time.time()
    train_loss, train_acc = train_epoch(classifier, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(classifier, val_loader, criterion)
    print("")
    print(f'Epoch: {epoch+1:02} | Time: {time.time() - start_time:.0f}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

test_loss, test_acc = evaluate(classifier, test_loader, criterion)
print("")
print(f'Test Loss: {test_loss:.3f} |  Test Acc: {test_acc*100:.2f}%')


Epoch: 01 | Time: 29s
	Train Loss: 1.460 | Train Acc: 41.83%
	 Val. Loss: 1.088 |  Val. Acc: 60.50%



Epoch: 02 | Time: 29s
	Train Loss: 0.774 | Train Acc: 72.44%
	 Val. Loss: 0.591 |  Val. Acc: 78.95%



Test Loss: 0.663 |  Test Acc: 75.80%


💡 **Résultat.** Le gain est énorme (dans notre exécution : 79 % en validation et 76 % en test, contre ~39 % et 38 % avec `lr = 0.001`) pour un seul paramètre changé. Avec un pas plus petit, les poids s'ajustent progressivement et le modèle apprend vraiment à utiliser les mots de la phrase.

Et le sur-apprentissage ? Si l'accuracy de validation reste **supérieure ou égale** à celle du train, il n'y en a pas encore. (La validation peut même être meilleure que le train : l'accuracy de train est une moyenne sur toute l'epoch, y compris le début, quand le modèle était encore mauvais.) On pourrait donc ajouter quelques epochs en surveillant la loss de validation, et s'arrêter quand elle remonte (*early stopping*).

**Leçon :** quand un modèle « n'apprend rien », le premier suspect est le **learning rate**, avant même l'architecture.

Les prédictions ci-dessous utilisent ce modèle amélioré.

**Exercise.** Try out the model in a bunch of sentences of your own.

💡 **Comment prédire sur une nouvelle phrase ?** On refait exactement le même traitement que pour les données d'entraînement :
1. tokenizer, qui ajoute `[CLS]` et `[SEP]`, puis conversion en tenseur `(1, s)` (un batch d'une seule phrase) ;
2. passage dans le modèle en mode `eval` et sans gradient, ce qui donne 6 logits ;
3. `softmax` pour obtenir des probabilités lisibles, puis `argmax` pour l'émotion prédite.

In [34]:
def predict(sentence):
    classifier.eval()
    inputs = tokenizer(sentence, truncation=True, return_tensors="pt")  # return_tensors="pt" -> tenseurs PyTorch
    with torch.no_grad():
        logits = classifier(inputs["input_ids"].to(device))
    probs = torch.softmax(logits, dim=-1)[0]  # 6 probabilités qui somment à 1
    best = probs.argmax().item()
    return label_names[best], probs[best].item()


sentences = [
    "I feel so happy and grateful today",
    "I am terrified of what might happen tomorrow",
    "I miss you so much, I feel lonely without you",
    "I can't believe they lied to me again, I'm furious",
    "I love spending time with my family",
    "I was shocked when I saw the results",
    "I didn't enjoy the movie at all",
]

for sentence in sentences:
    emotion, confidence = predict(sentence)
    print(f"{emotion:9s} ({confidence:.0%})  {sentence}")

joy       (93%)  I feel so happy and grateful today
fear      (99%)  I am terrified of what might happen tomorrow
sadness   (66%)  I miss you so much, I feel lonely without you
anger     (38%)  I can't believe they lied to me again, I'm furious
joy       (94%)  I love spending time with my family
surprise  (96%)  I was shocked when I saw the results
joy       (81%)  I didn't enjoy the movie at all


💡 **Ce qu'on observe** (avec le modèle entraîné à `lr = 0.0001`) :
- les émotions marquées par un mot-clé clair sont bien reconnues, avec une forte confiance : *happy* → joy (93 %), *terrified* → fear (99 %), *shocked* → surprise (96 %), *lonely* → sadness ;
- *furious* → anger est juste, mais avec seulement 38 % de confiance : le modèle hésite ;
- *"I love spending time with my family"* donne **joy** et non *love*. *Love* est une classe rare (8 % des données) et très proche de *joy* ;
- *"I didn't enjoy the movie at all"* donne **joy à 81 %** : le modèle a vu *enjoy* et a ignoré la **négation**. Avec un seul bloc et sans pré-entraînement, il s'appuie surtout sur la présence de certains mots, pas sur le sens de la phrase entière.

Ces limites expliquent l'intérêt du **pré-entraînement** sur d'énormes corpus (BERT, Notebook 2) : le modèle arrive en sachant déjà comment fonctionne l'anglais, y compris les négations.